# Chapter 17: Creating Pivot Tables and Cross-Tabulations for Summarizing and Comparing Data

**Companion notebook** for *Beginner's Guide to Pandas* by Ravi Shankar

Run each cell in order. Exercises are at the end.

In [1]:
import pandas as pd
import numpy as np

# Creating Pivot Tables and Cross-Tabulations for Summarizing and Comparing Data

Pivot tables and cross-tabulations are powerful tools for transforming raw data into meaningful summaries. They allow you to reorganize, aggregate, and compare data across multiple dimensions, making it easier to identify patterns and trends.

---

## What Are Pivot Tables?

When working with real-world datasets, raw tabular data can be difficult to interpret at a glance. Pivot tables and cross-tabulations are powerful summarization tools that reorganize and aggregate data, making patterns and comparisons immediately visible. If you have ever used pivot tables in spreadsheet software like Excel, pandas offers the same capability — with far greater flexibility and programmability.

In this chapter, you will learn how to use `pd.pivot_table()` and `pd.crosstab()` to summarize, group, and compare data across multiple dimensions. This first section establishes the foundational concepts and data structures you will build on throughout the chapter.

---

## Setting Up: Importing pandas and Creating Sample Data

Before building any pivot table, you need a DataFrame to work with. The example below creates a small sales dataset — a format you will encounter frequently in business analytics contexts.

> *If you are new to pandas, `pd.DataFrame(data)` converts a Python dictionary into a tabular DataFrame. Each dictionary key becomes a column name, and its list of values becomes the column's data.*

In [2]:
import pandas as pd
import numpy as np

# Sample sales data: region, product, salesperson, and revenue
data = {
    "region":      ["North", "North", "South", "South", "East", "East"],
    "product":     ["Widget", "Gadget", "Widget", "Gadget", "Widget", "Gadget"],
    "salesperson": ["Alice", "Bob",    "Alice",  "Bob",   "Alice",  "Bob"],
    "revenue":     [200,      150,      300,      100,     250,      175]
}

df = pd.DataFrame(data)
print(df)

  region product salesperson  revenue
0  North  Widget       Alice      200
1  North  Gadget         Bob      150
2  South  Widget       Alice      300
3  South  Gadget         Bob      100
4   East  Widget       Alice      250
5   East  Gadget         Bob      175


**Output:**
```
  region product salesperson  revenue
0  North  Widget       Alice      200
1  North  Gadget         Bob      150
2  South  Widget       Alice      300
3  South  Gadget         Bob      100
4   East  Widget       Alice      250
5   East  Gadget         Bob      175
```

Each row represents a single sales transaction. Notice that the data is **flat** — every observation occupies its own row, and relationships between categories are not immediately obvious. This is exactly the kind of data that pivot tables are designed to clarify.

---

## Core Concept: From Raw Data to Summary Table

A pivot table takes a **flat, row-oriented dataset** and restructures it so that:

- **Rows** represent one categorical variable
- **Columns** represent another categorical variable
- **Cell values** represent an aggregated metric (e.g., sum, mean, count)

For example, raw sales data with one row per transaction:

| Region | Product | Revenue |
|--------|---------|---------|
| East   | Widget  | 100     |
| West   | Gadget  | 200     |
| East   | Gadget  | 150     |
| West   | Widget  | 120     |

...becomes a summary table after pivoting:

| Region | Gadget | Widget |
|--------|--------|--------|
| East   | 150    | 100    |
| West   | 200    | 120    |

Each cell now holds an aggregated value (here, the sum of Revenue) for that Region–Product combination. This is the core transformation a pivot table performs.

---

## Your First Pivot Table

Now that the data is loaded, use `pd.pivot_table()` to summarize total revenue by region and product. The key parameters are:

- `values`: the column to aggregate (`"revenue"`)
- `index`: the row grouping variable (`"region"`)
- `columns`: the column grouping variable (`"product"`)
- `aggfunc`: the aggregation function (`"sum"`)

In [3]:
# Summarize total revenue by region (rows) and product (columns)
pivot = pd.pivot_table(
    df,
    values="revenue",
    index="region",
    columns="product",
    aggfunc="sum"
)

print(pivot)

product  Gadget  Widget
region                 
East        175     250
North       150     200
South       100     300


**Output:**
```
product  Gadget  Widget
region
East        175     250
North       150     200
South       100     300
```

The six-row dataset has been condensed into a compact 3×2 summary table. Each cell now holds the total revenue for a specific region–product combination. For example, the South region generated **$300** in Widget sales and **$100** in Gadget sales. Patterns that were hidden in the raw data — such as Widgets consistently outperforming Gadgets — are now immediately apparent.

---

## Key Takeaways (Section 1)

| Concept | Description |
|---|---|
| **Flat data** | Raw row-per-observation format; relationships are implicit |
| **Pivot table** | Restructured summary with rows, columns, and aggregated values |
| `pd.pivot_table()` | The primary pandas function for creating pivot tables |
| `values` | The column whose values are aggregated |
| `index` | The column that becomes row labels |
| `columns` | The column that becomes column headers |
| `aggfunc` | Controls how values are combined (sum, mean, count, etc.) |

In the sections that follow, you will progressively add complexity: handling multiple aggregation functions, working with missing values, adding margins (subtotals), and ultimately building cross-tabulations for categorical frequency analysis.

---

## Understanding the Foundation — What Are Pivot Tables?

Before writing any code, it helps to understand what a pivot table actually does conceptually. A pivot table **reorganizes ("pivots") rows of raw data into a structured summary**, grouping values by one or more categories and applying an aggregation function (such as sum, mean, or count) to a numeric column. Think of it as answering the question: *"For each combination of categories, what is the total/average/count of this measurement?"*

Consider a simple sales dataset where each row records a single transaction. A pivot table can instantly answer: *"What were the total sales for each product, broken down by region?"* — collapsing hundreds of rows into a compact, readable grid.

---

## The Mental Model: From Long to Wide

A useful way to think about this transformation is the shift from **long format** to **wide format**. In **long format**, each row represents a single observation, with category information stored in columns. In **wide format**, category combinations become the row and column labels, and each cell holds a computed value:

```
LONG FORMAT (raw data)          WIDE FORMAT (pivot table)
─────────────────────           ─────────────────────────
region  product  sales          product  Gadget  Widget
North   Widget   150            region
South   Widget   200            North       300     250
North   Gadget   300    ──►     South       650     200
South   Gadget   250
North   Widget   100
South   Gadget   400
```

This restructuring is the core operation you'll build on throughout this chapter. In later sections, we'll add multiple index levels, multiple aggregation functions, and margin totals — but every technique extends this same fundamental idea.

---

## The Raw Data We'll Work With

Throughout this section, we'll use a small but realistic sales dataset. Let's create it first so you can see exactly what we're starting with.

In [4]:
import pandas as pd

# A flat table of individual sales transactions
data = {
    "region":  ["North", "South", "North", "South", "North", "South"],
    "product": ["Widget", "Widget", "Gadget", "Gadget", "Widget", "Gadget"],
    "sales":   [150, 200, 300, 250, 100, 400],
}

df = pd.DataFrame(data)
print(df)

  region product  sales
0  North  Widget    150
1  South  Widget    200
2  North  Gadget    300
3  South  Gadget    250
4  North  Widget    100
5  South  Gadget    400


**Output:**
```
  region product  sales
0  North  Widget    150
1  South  Widget    200
2  North  Gadget    300
3  South  Gadget    250
4  North  Widget    100
5  South  Gadget    400
```

Each row represents one transaction. Notice that the same region–product combinations appear multiple times (e.g., North/Widget appears in rows 0 and 4). This repetition is exactly what a pivot table is designed to consolidate.

---

## What Makes a Good Pivot Table Dataset

A pivot table works best when your data is in **tidy format**: one row per observation, with separate columns for categories, subcategories, and numeric values. The function then lets you promote any categorical column to a row index or column header, and aggregate the numeric values where those categories intersect.

---

## The Core `pivot_table()` Signature

The four essential parameters map directly to the four questions a pivot table answers:

In [5]:
import pandas as pd

# A minimal sales dataset in tidy format
data = {
    'region':  ['North', 'North', 'South', 'South', 'North', 'South'],
    'product': ['Widget', 'Gadget', 'Widget', 'Gadget', 'Widget', 'Gadget'],
    'quarter': ['Q1', 'Q1', 'Q1', 'Q1', 'Q2', 'Q2'],
    'revenue': [1200, 850, 940, 1100, 1350, 980]
}
df = pd.DataFrame(data)
print(df)

  region product quarter  revenue
0  North  Widget      Q1     1200
1  North  Gadget      Q1      850
2  South  Widget      Q1      940
3  South  Gadget      Q1     1100
4  North  Widget      Q2     1350
5  South  Gadget      Q2      980


```
  region product quarter  revenue
0  North  Widget      Q1     1200
1  North  Gadget      Q1      850
2  South  Widget      Q1      940
3  South  Gadget      Q1     1100
4  North  Widget      Q2     1350
5  South  Gadget      Q2      980
```

The raw data has six individual transactions. Each row answers *who sold what, when, and for how much* — but spotting patterns requires scanning every row manually.

---

## Building the Simplest Pivot Table

In [6]:
# Minimum viable pivot table:
# values  → what to aggregate
# index   → which column becomes the row labels
# columns → which column becomes the column headers
# aggfunc → how to combine multiple values at each intersection

simple_pivot = pd.pivot_table(
    df,
    values='revenue',
    index='region',
    columns='product',
    aggfunc='sum'
)
print(simple_pivot)

product  Gadget  Widget
region                 
North       850    2550
South      2080     940


```
product  Gadget  Widget
region
North       850    2550
South      2080     940
```

Each cell holds the **sum of revenue** at the intersection of one region and one product. North sold two Widget transactions (1200 + 1350 = 2550), while South sold two Gadget transactions (1100 + 980 = 2080). The raw six rows have been compressed into a four-cell summary that makes regional and product differences immediately visible.

---

## What Happens at Each Intersection

Understanding how pandas fills each cell prevents confusion later when data is sparse or when multiple rows map to the same intersection.

In [7]:
# Add a duplicate intersection to see aggregation in action
data_with_duplicate = {
    'region':  ['North', 'North', 'North'],
    'product': ['Widget', 'Widget', 'Gadget'],
    'revenue': [1200, 1350, 850]
}
df_dup = pd.DataFrame(data_with_duplicate)

# pandas finds all rows where region='North' AND product='Widget',
# then applies aggfunc to those revenue values
pivot_dup = pd.pivot_table(
    df_dup,
    values='revenue',
    index='region',
    columns='product',
    aggfunc='sum'
)
print(pivot_dup)
print(f"\nManual check — North/Widget sum: {1200 + 1350}")

product  Gadget  Widget
region                 
North       850    2550

Manual check — North/Widget sum: 2550


```
product  Gadget  Widget
region
North       850    2550

Manual check — North/Widget sum: 2550
```

The North/Widget cell aggregates both matching rows (1200 + 1350) into a single value. This grouping-then-aggregating logic is the engine behind every pivot table, no matter how complex. Sections that follow will layer on multiple index levels, multiple value columns, and custom aggregation functions — all of which extend this same core mechanism.

---

## Building Your First Pivot Table

Let's start with a practical example using sales data that spans multiple regions, quarters, and products:

In [8]:
import pandas as pd
import numpy as np

# Create sample sales data with three categorical dimensions
# and one numeric measure we want to summarize
sales_data = pd.DataFrame({
    'Region': ['North', 'North', 'North', 'South', 'South', 'South',
               'East', 'East', 'East', 'West', 'West', 'West'],
    'Quarter': ['Q1', 'Q2', 'Q3', 'Q1', 'Q2', 'Q3',
                'Q1', 'Q2', 'Q3', 'Q1', 'Q2', 'Q3'],
    'Product': ['A', 'A', 'B', 'A', 'B', 'B',
                'A', 'B', 'A', 'B', 'A', 'B'],
    'Revenue': [15000, 18000, 12000, 22000, 19000, 21000,
                17000, 16000, 20000, 14000, 18000, 19000]
})

print(sales_data)

   Region Quarter Product  Revenue
0   North      Q1       A    15000
1   North      Q2       A    18000
2   North      Q3       B    12000
3   South      Q1       A    22000
4   South      Q2       B    19000
5   South      Q3       B    21000
6    East      Q1       A    17000
7    East      Q2       B    16000
8    East      Q3       A    20000
9    West      Q1       B    14000
10   West      Q2       A    18000
11   West      Q3       B    19000


```
   Region Quarter Product  Revenue
0   North      Q1       A    15000
1   North      Q2       A    18000
2   North      Q3       B    12000
3   South      Q1       A    22000
4   South      Q2       B    19000
5   South      Q3       B    21000
6    East      Q1       A    17000
7    East      Q2       B    16000
8    East      Q3       A    20000
9    West      Q1       B    14000
10   West      Q2       A    18000
11   West      Q3       B    19000
```

In this flat format, comparing regions across quarters requires scanning multiple rows. A pivot table collapses this into a grid that makes comparisons immediate. The four key parameters of `pd.pivot_table()` map directly to that grid structure: `values` specifies what to measure, `index` sets the row labels, `columns` sets the column labels, and `aggfunc` defines how to combine multiple values that fall into the same cell.

In [9]:
# Simple pivot table — mean revenue by Region (rows) and Quarter (columns)
# Each cell shows the average Revenue for that Region/Quarter combination
pivot = pd.pivot_table(
    sales_data,
    values='Revenue',
    index='Region',
    columns='Quarter',
    aggfunc='mean'
)

print(pivot)

Quarter       Q1       Q2       Q3
Region                            
East     17000.0  16000.0  20000.0
North    15000.0  18000.0  12000.0
South    22000.0  19000.0  21000.0
West     14000.0  18000.0  19000.0


```
Quarter      Q1      Q2      Q3
Region
East    17000.0 16000.0 20000.0
North   15000.0 18000.0 12000.0
South   22000.0 19000.0 21000.0
West    14000.0 18000.0 19000.0
```

The output immediately reveals patterns that were invisible in the raw data. Regions are sorted alphabetically along the rows, quarters appear as labeled columns, and each cell holds the mean revenue for that intersection. Notice that pandas uses `float64` for the values (hence the `.0` suffix) even though the source data was integers — this is standard behavior when `aggfunc='mean'` is applied, since averages are not guaranteed to be whole numbers.

In this dataset each Region/Quarter combination has exactly one row, so `mean` returns the original value unchanged. In the next section you'll see how `aggfunc` becomes essential when multiple records map to the same cell.

---

## Aggregation Functions: Beyond Simple Averages

While the default pivot table uses the mean, real-world analysis often requires multiple statistical measures simultaneously. Pandas supports passing a list of aggregation functions or mapping specific functions to specific columns, giving you fine-grained control over how each value is summarised.

### Applying Multiple Functions to a Single Column

When you need more than one statistic for the same field — for example, both the total and the average revenue per region and quarter — pass a **list** of function names to `aggfunc`.

> **About the result structure:** When `aggfunc` receives a list, pandas produces a **MultiLevel column index** (also called a hierarchical index). Think of it as two rows of column labels stacked on top of each other: the outer row names the statistic (`sum`, `mean`, `count`) and the inner row names the original column category (here, the quarter). You can slice a single statistic out of the result with standard bracket notation — for example, `pivot_multi['sum']` — which returns a plain DataFrame containing only the totals.
>
> Note: `'count'` tallies **non-null values** in each group, not total rows, so it can differ from `'sum'` if missing data is present (covered in the next section).

In [10]:
import pandas as pd

# Sample sales dataset used throughout this chapter
sales_data = pd.DataFrame({
    'Region':  ['North', 'North', 'South', 'South', 'North', 'South'],
    'Quarter': ['Q1',    'Q2',    'Q1',    'Q2',    'Q1',    'Q2'],
    'Product': ['A', 'B', 'A', 'B', 'B', 'A'],
    'Revenue': [1200, 1500, 900, 1100, 1350, 950],
    'Units':   [30,   40,   25,  28,   35,   22]
})

# Apply sum, mean, and count to Revenue across Region × Quarter
pivot_multi = pd.pivot_table(
    sales_data,
    values='Revenue',
    index='Region',
    columns='Quarter',
    aggfunc=['sum', 'mean', 'count']
)

print(pivot_multi)

          sum          mean         count   
Quarter    Q1    Q2      Q1      Q2    Q1 Q2
Region                                      
North    2550  1500  1275.0  1500.0     2  1
South     900  2050   900.0  1025.0     1  2


```
        sum          mean         count
Quarter  Q1     Q2    Q1     Q2    Q1  Q2
Region
North  2550   1500  1275.0  1500.0   2   1
South   900   2050   900.0  1025.0   1   2
```

The outermost column level is the function name; the inner level is the quarter. To work with just one statistic, slice by its name:

In [11]:
# Extract only the sum subtable — result is a standard DataFrame
print(pivot_multi['sum'])

Quarter    Q1    Q2
Region             
North    2550  1500
South     900  2050


```
Quarter    Q1    Q2
Region
North    2550  1500
South     900  2050
```

### Mapping Different Functions to Different Columns

As your analysis grows more complex, you may want distinct statistics for each column. Pass a dictionary to `aggfunc` where each key is a column name and each value is a function or list of functions:

In [12]:
# Revenue: total, average, and variability; Units: total only
pivot_custom = pd.pivot_table(
    sales_data,
    values=['Revenue', 'Units'],
    index=['Region', 'Product'],
    aggfunc={'Revenue': ['sum', 'mean', 'std'],
             'Units':   'sum'}
)

print(pivot_custom)

               Revenue                   Units
                  mean         std   sum   sum
Region Product                                
North  A        1200.0         NaN  1200    30
       B        1425.0  106.066017  2850    75
South  A         925.0   35.355339  1850    47
       B        1100.0         NaN  1100    28


```
                Revenue                       Units
                   mean    std     sum          sum
Region Product
North  A         1200.0    NaN    1200           30
       B         1425.0  106.07   2850           75
South  A          925.0   35.36   1850           47
       B         1100.0    NaN    1100           28
```

> **Understanding the hierarchical row index:** Passing a list to `index` (here `['Region', 'Product']`) produces a **MultiIndex** — a two-level row label where each `Region` groups its `Product` rows beneath it. You can read the output top-to-bottom: North contains rows A and B; South contains rows A and B. This nesting will appear throughout the rest of the chapter.

Several details are worth noting:

- **`std` returns `NaN`** for groups with only one observation (North–A, South–B) because standard deviation is undefined for a single value — a useful data-quality signal. The upcoming **"Handling Missing Values"** section covers strategies for filling or dropping these results.
- **Column ordering** in the output follows the function list you provided, not alphabetical order.
- The **hierarchical row index** (`Region` → `Product`) mirrors the list passed to `index`, establishing a pattern you will extend in the **"Adding Margins"** and **"Handling Missing Values"** sections later in this chapter.

---

## Handling Missing Values

Real-world datasets frequently contain gaps — not every region sells every product, and not every time period has recorded transactions. When a pivot table encounters these gaps, it produces `NaN` (Not a Number — a special floating-point sentinel that pandas uses to represent absent or undefined data) by default, which can complicate downstream calculations. Pandas provides a built-in parameter to handle this gracefully.

To see why this matters, first observe what a pivot table looks like *without* any missing value handling:

In [13]:
# Default behavior: missing combinations appear as NaN
pivot_default = pd.pivot_table(
    sales_data,
    values='Revenue',
    index='Region',
    columns='Product',
    aggfunc='sum'
)

print(pivot_default)

Product     A     B
Region             
North    1200  2850
South    1850  1100


This produces output where sparse region-product combinations show `NaN`, making aggregations like column totals unreliable. Now apply the `fill_value` parameter to replace those gaps with a meaningful substitute:

In [14]:
# Replace NaN entries with 0 to indicate no sales occurred
pivot_filled = pd.pivot_table(
    sales_data,
    values='Revenue',
    index='Region',
    columns='Product',
    aggfunc='sum',
    fill_value=0
)

print(pivot_filled)

Product     A     B
Region             
North    1200  2850
South    1850  1100


The output now shows `0` wherever a region-product combination had no recorded sales, rather than `NaN`. This distinction is important: `0` signals *confirmed absence of revenue*, while `NaN` signals *unknown or unrecorded data*. Choose `fill_value` deliberately based on what the missing data actually represents in your domain.

In the next section you will see how combining `fill_value` with `margins=True` ensures that row and column totals calculate correctly — because no `NaN` values remain to silently distort the sums.

---

## Adding Margins (Subtotals and Grand Totals)

When analyzing pivot tables, you often need row and column totals to understand both individual breakdowns and overall performance. The `margins` parameter in `pd.pivot_table()` automatically appends these aggregate totals, saving you from computing them manually.

In [15]:
# Add margins for totals
pivot_margins = pd.pivot_table(
    sales_data,
    values='Revenue',
    index='Region',
    columns='Quarter',
    aggfunc='sum',
    margins=True,         # Enables row and column totals
    margins_name='Total'  # Labels the totals row/column (default is 'All')
)

print(pivot_margins)

Quarter    Q1    Q2  Total
Region                    
North    2550  1500   4050
South     900  2050   2950
Total    3450  3550   7000


Output:
```
Quarter      Q1      Q2      Q3    Total
Region
East      17000   16000   20000   53000
North     15000   18000   12000   45000
South     22000   19000   21000   62000
West      14000   18000   19000   51000
Total     68000   71000   72000  211000
```

The `margins=True` parameter adds two things simultaneously: a **`Total` column** on the right (summing each region's revenue across all quarters) and a **`Total` row** at the bottom (summing each quarter's revenue across all regions). The bottom-right cell (`211000`) is the grand total — the sum of all revenue across every region and quarter combined.

Notice that `margins_name='Total'` replaces the default label `'All'` with something more descriptive. This label appears in both the appended row and column, so choosing a meaningful name improves readability when sharing reports.

---

## Working with Categorical Data

Before diving into cross-tabulations, it is worth understanding how pandas represents categorical data, because cross-tabulations are almost exclusively applied to categorical columns.

### The pandas `Categorical` Dtype

Pandas provides a dedicated `Categorical` dtype for columns whose values come from a fixed, known set of possibilities. Unlike a plain string column — where pandas only knows about values that are actually present — a `Categorical` column carries an explicit list of all *valid* values, called its **categories**, regardless of whether every category appears in the data.

You create a categorical column with `pd.Categorical()`:

In [17]:
values = ['Q1', 'Q2', 'Q3', 'Q1', 'Q2']
pd.Categorical(values, categories=None, ordered=False)

['Q1', 'Q2', 'Q3', 'Q1', 'Q2']
Categories (3, object): ['Q1', 'Q2', 'Q3']

- **`values`** — the actual data (a list or array).
- **`categories`** — the complete universe of valid values. Any value in `categories` but absent from `values` is still a recognised level.
- **`ordered`** — set to `True` if the categories have a meaningful order (e.g., `Low < Medium < High`).

Once a `Series` has `Categorical` dtype, you access its categorical properties through the **`.cat` accessor**. The most commonly used attributes are:

| Attribute | What it returns |
|---|---|
| `.cat.categories` | An `Index` of all defined category levels |
| `.cat.codes` | Integer codes mapping each value to its position in `.cat.categories` |
| `.cat.ordered` | `True` if the categories are ordered |

For cross-tabulations, the categorical dtype matters because pandas will include **all defined categories** in the output table, even if some combinations have a count of zero — giving you a complete picture rather than a sparse one.

---

## Cross-Tabulations for Categorical Analysis

Cross-tabulations (also called contingency tables) count how often combinations of categorical values co-occur in your data. Unlike pivot tables, which aggregate numerical values, cross-tabulations focus purely on the **frequency of categorical relationships** — making them ideal for understanding how two or more categorical variables interact.

The primary tool is `pd.crosstab()`:

In [19]:
import pandas as pd

# Example data
df = pd.DataFrame({
    'department': ['Sales', 'IT', 'Sales', 'HR', 'IT'],
    'gender': ['M', 'F', 'F', 'M', 'M'],
    'salary': [50000, 70000, 55000, 60000, 75000]
})

# Define variables used by crosstab
index = df['department']
columns = df['gender']

result = pd.crosstab(
    index=index,              # Row variable(s)
    columns=columns,          # Column variable(s)
    values=None,              # Optional column to aggregate
    aggfunc=None,             # Required when values is provided
    rownames=['Department'],  # Row label
    colnames=['Gender'],      # Column label
    margins=False,            # Add totals if True
    margins_name="All",
    normalize=False,          # Convert counts to proportions if desired
    dropna=True
)

print(result)

Gender      F  M
Department      
HR          0  1
IT          1  1
Sales       1  1


The key distinction from `pivot_table()`:

| Feature | `pivot_table()` | `pd.crosstab()` |
|---|---|---|
| Primary use | Aggregate numeric values | Count categorical co-occurrences |
| Default aggregation | `mean` | `count` (frequency) |
| Input | DataFrame columns by name | Series objects directly |
| Normalization built-in | No | Yes (`normalize=`) |

The subsections that follow walk through each capability in turn, building from a simple frequency table up to weighted and multi-level cross-tabulations.

### Basic Frequency Cross-Tabulation

Start with the simplest case: counting how many times each Region–Product combination appears in the dataset.

In [20]:
import pandas as pd

# Sample sales dataset with categorical variables
sales_data = pd.DataFrame({
    'Region':  ['East', 'East', 'East', 'North', 'North', 'North',
                'South', 'South', 'South', 'West', 'West', 'West'],
    'Product': ['A', 'A', 'B', 'A', 'A', 'B',
                'A', 'B', 'B', 'A', 'A', 'B'],
    'Sales':   [200, 150, 300, 250, 180, 220,
                170, 310, 290, 240, 195, 275]
})

# Basic cross-tabulation: counts occurrences of each Region-Product pair
crosstab = pd.crosstab(
    sales_data['Region'],
    sales_data['Product']
)

print(crosstab)

Product  A  B
Region       
East     2  1
North    2  1
South    1  2
West     2  1


Output:
```
Product  A  B
Region
East     2  1
North    2  1
South    1  2
West     2  1
```

Each cell shows how many rows in `sales_data` share that Region–Product combination. Notice that `pd.crosstab()` automatically labels the axes using the Series names (`Region` and `Product`), keeping the output self-documenting.

### Adding Margins for Row and Column Totals

Raw counts become more meaningful when you can see them in context of row and column totals. You already saw `margins=True` used with `pivot_table()` earlier in this chapter; `pd.crosstab()` supports the same parameter. It appends an `All` row and column — and the companion `margins_name=` parameter lets you replace that default label with something more descriptive:

In [21]:
# Cross-tabulation with row and column totals
crosstab_margins = pd.crosstab(
    sales_data['Region'],
    sales_data['Product'],
    margins=True,
    margins_name='Total'   # rename the default 'All' label
)

print(crosstab_margins)

Product  A  B  Total
Region              
East     2  1      3
North    2  1      3
South    1  2      3
West     2  1      3
Total    7  5     12


Output:
```
Product  A  B  Total
Region
East     2  1      3
North    2  1      3
South    1  2      3
West     2  1      3
Total    7  5     12
```

The `Total` column confirms each region has exactly 3 records, while the `Total` row shows Product A appears 7 times overall versus 5 for Product B — a distribution that would be invisible in the basic table.

### Normalizing to Reveal Proportions

Absolute counts can be misleading when group sizes differ. Normalizing converts counts to proportions, making comparisons fair across groups. The `normalize` parameter accepts three values:

| Value | Divides by |
|-------|-----------|
| `'index'` | Each row total |
| `'columns'` | Each column total |
| `True` or `'all'` | Grand total |

In [22]:
# Normalize by row: what share of each region's sales belong to each product?
crosstab_row_pct = pd.crosstab(
    sales_data['Region'],
    sales_data['Product'],
    normalize='index'       # proportions sum to 1.0 across each row
).round(2)                  # .round(2) rounds all values to 2 decimal places

print("Row-normalized (proportion of each region's records per product):")
print(crosstab_row_pct)

# Normalize by column: what share of each product's sales come from each region?
crosstab_col_pct = pd.crosstab(
    sales_data['Region'],
    sales_data['Product'],
    normalize='columns'     # proportions sum to 1.0 down each column
).round(2)

print("\nColumn-normalized (proportion of each product's records per region):")
print(crosstab_col_pct)

Row-normalized (proportion of each region's records per product):
Product     A     B
Region             
East     0.67  0.33
North    0.67  0.33
South    0.33  0.67
West     0.67  0.33

Column-normalized (proportion of each product's records per region):
Product     A    B
Region            
East     0.29  0.2
North    0.29  0.2
South    0.14  0.4
West     0.29  0.2


Output:
```
Row-normalized (proportion of each region's records per product):
Product     A     B
Region
East     0.67  0.33
North    0.67  0.33
South    0.33  0.67
West     0.67  0.33

Column-normalized (proportion of each product's records per region):
Product     A     B
Region
East     0.29  0.20
North    0.29  0.20
South    0.14  0.40
West     0.29  0.20
```

> **Note on `.round()`:** `DataFrame.round(n)` is a standard pandas method that rounds every value in the table to `n` decimal places. It is used here purely for display clarity and does not affect the underlying data.

Row normalization immediately highlights that South is the only region where Product B outsells Product A (67% vs. 33%). Column normalization reveals that South accounts for 40% of all Product B records but only 14% of Product A — a pattern that points to a genuine regional preference worth investigating further.

### Multi-Level Cross-Tabulations

Building on the two-dimensional cross-tabulations from previous sections, you can extend the analysis to examine relationships across three or more dimensions simultaneously. Multi-level cross-tabulations nest grouping variables, revealing patterns that would be invisible in simpler summaries.

To create a multi-level cross-tabulation, pass a list of Series as the `index` argument. Each unique combination of values across those Series becomes a row in the resulting table:

In [23]:
import pandas as pd
import numpy as np

# Reproducible sample dataset
np.random.seed(42)
n = 200

sales_data = pd.DataFrame({
    'Region':  np.random.choice(['North', 'South', 'East', 'West'], n),
    'Quarter': np.random.choice(['Q1', 'Q2', 'Q3', 'Q4'], n),
    'Product': np.random.choice(['Widget', 'Gadget', 'Doohickey'], n),
})

# Cross-tabulation with multiple row variables and column totals
crosstab_multi = pd.crosstab(
    index=[sales_data['Region'], sales_data['Quarter']],  # nested row levels
    columns=sales_data['Product'],                        # column variable
    margins=True                                          # add row/column totals
)

print(crosstab_multi)

Product         Doohickey  Gadget  Widget  All
Region Quarter                                
East   Q1               2       5       6   13
       Q2               3       4       5   12
       Q3               7       6       4   17
       Q4               3       3       6   12
North  Q1               3       3       6   12
       Q2               2       4       1    7
       Q3               2       3       4    9
       Q4               8       7       3   18
South  Q1               5       3       3   11
       Q2               6       0       3    9
       Q3               2       5       3   10
       Q4               1      11       4   16
West   Q1               3       4       6   13
       Q2               3       3       6   12
       Q3               4       5       5   14
       Q4               4       7       4   15
All                    58      73      69  200


```
Product          Doohickey  Gadget  Widget  All
Region Quarter
East   Q1                3       4       3   10
       Q2                2       3       4    9
       Q3                4       2       3    9
       Q4                3       5       2   10
North  Q1                2       3       5   10
...
All                      65      70      65  200
```

The output is a hierarchically indexed DataFrame. The two leftmost levels — **Region** and **Quarter** — form a `MultiIndex` (pandas' structure for hierarchical, multi-level row labels) on the rows, while each product occupies its own column. Reading across a single row (e.g., `East / Q1`) shows how sales were distributed among products for that specific region-quarter combination. The `All` column and row (added by `margins=True`) provide marginal totals, making it straightforward to compare a cell count against its row total, column total, or grand total.

### Weighted Cross-Tabulations

Building on the frequency-based cross-tabulations from previous sections, you can extend `pd.crosstab()` beyond simple counts by supplying a `values` parameter alongside an aggregation function. This transforms the table from tallying occurrences into summarising a numeric measure — such as total revenue, average units sold, or maximum discount — across each combination of categorical variables.

> **Note:** `values` and `aggfunc` must always be supplied together. Providing `values` without `aggfunc` raises a `ValueError` — pandas requires you to specify *how* to aggregate the values you have pointed it to.

In [24]:
import pandas as pd
import numpy as np

# Sample sales dataset with region, product, and revenue columns
sales_data = pd.DataFrame({
    'Region':  ['North', 'North', 'South', 'South', 'East', 'East',
                'North', 'South', 'East',  'North'],
    'Product': ['Widget', 'Gadget', 'Widget', 'Gadget', 'Widget', 'Gadget',
                'Widget', 'Widget', 'Gadget', 'Gadget'],
    'Revenue': [1200, 850, 950, 1100, 780, 920, 1350, 870, 1050, 760]
})

# Cross-tabulation aggregating revenue with sum
crosstab_agg = pd.crosstab(
    sales_data['Region'],
    sales_data['Product'],
    values=sales_data['Revenue'],
    aggfunc='sum'
)

print(crosstab_agg)

Product  Gadget  Widget
Region                 
East       1970     780
North      1610    2550
South      1100    1820


```
Product  Gadget  Widget
Region
East       1970     780
North      1610    2550
South      1100    1820
```

Each cell now contains the **total revenue** for that region–product combination rather than a count of transactions. You can swap `aggfunc='sum'` for other functions to answer different questions:

In [25]:
# Cross-tabulation aggregating revenue with mean, rounded for readability
crosstab_mean = pd.crosstab(
    sales_data['Region'],
    sales_data['Product'],
    values=sales_data['Revenue'],
    aggfunc='mean'
).round(1)

print(crosstab_mean)

Product  Gadget  Widget
Region                 
East      985.0   780.0
North     805.0  1275.0
South    1100.0   910.0


```
Product  Gadget   Widget
Region
East     985.0    780.0
North    805.0   1275.0
South   1100.0    910.0
```

> **Note:** When `values` is specified, any cell with no matching rows will contain `NaN` rather than `0`. If downstream calculations require zero-filled tables, chain `.fillna(0)` onto the result.

---

## Working with Categorical Data in Cross-Tabulations

### The `dropna` Parameter of `pd.crosstab()`

By default, `pd.crosstab()` silently ignores any category level that has no matching rows in the data (`dropna=True`). Setting `dropna=False` overrides this: every level defined on a `Categorical` column will appear in the output, with unobserved cells filled with `0`.

This distinction only matters when at least one of the input columns is a `Categorical` with more defined categories than are present in the data. For plain string or integer columns, `dropna` has no effect on which values appear.

First, construct a small DataFrame where the `Status` column has three defined categories (`Active`, `Inactive`, `Pending`) and `Department` has three categories (`Sales`, `IT`, `HR`), but only two rows of actual data exist:

In [26]:
import pandas as pd

# Create a DataFrame with categorical columns that have more defined
# categories than are actually present in the data
categories_df = pd.DataFrame({
    'Status': pd.Categorical(
        ['Active', 'Inactive'],
        categories=['Active', 'Inactive', 'Pending']
    ),
    'Department': pd.Categorical(
        ['Sales', 'IT'],
        categories=['Sales', 'IT', 'HR']
    ),
    'Count': [10, 5]
})

print("Source DataFrame:")
print(categories_df)
print(f"\nStatus categories : {categories_df['Status'].cat.categories.tolist()}")
print(f"Department categories: {categories_df['Department'].cat.categories.tolist()}")

Source DataFrame:
     Status Department  Count
0    Active      Sales     10
1  Inactive         IT      5

Status categories : ['Active', 'Inactive', 'Pending']
Department categories: ['Sales', 'IT', 'HR']


```
Source DataFrame:
     Status Department  Count
0    Active      Sales     10
1  Inactive         IT      5

Status categories : ['Active', 'Inactive', 'Pending']
Department categories: ['Sales', 'IT', 'HR']
```

The `.cat.categories` attribute confirms that `Pending` and `HR` are registered as valid categories even though no rows carry those values.

Now compare the default behaviour against `dropna=False`:

In [27]:
# Default: unobserved category combinations are dropped
crosstab_default = pd.crosstab(
    categories_df['Status'],
    categories_df['Department']
)

# dropna=False: all defined categories appear, unobserved cells filled with 0
crosstab_all = pd.crosstab(
    categories_df['Status'],
    categories_df['Department'],
    dropna=False
)

print("Default (dropna=True) — only observed categories shown:")
print(crosstab_default)

print("\nWith dropna=False — all defined categories retained:")
print(crosstab_all)

Default (dropna=True) — only observed categories shown:
Department  Sales  IT
Status               
Active          1   0
Inactive        0   1

With dropna=False — all defined categories retained:
Department  Sales  IT  HR
Status                   
Active          1   0   0
Inactive        0   1   0
Pending         0   0   0


```
Default (dropna=True) — only observed categories shown:
Department  IT  Sales
Status
Active       0     10
Inactive     5      0

With dropna=False — all defined categories retained:
Department  HR  IT  Sales
Status
Active       0   0     10
Inactive     0   5      0
Pending      0   0      0
```

Several things are worth noting:

- **`dropna=True` (default):** Only the two `Status` values and two `Department` values that actually appear in the data are included. The result is a compact 2×2 table.
- **`dropna=False`:** The full 3×3 grid is produced. `HR` appears as a column and `Pending` as a row, both filled with zeros. This is the correct choice whenever downstream code or a report template expects a fixed set of rows and columns regardless of data sparsity.
- **Zero vs. NaN:** Unobserved cells are filled with `0`, not `NaN`, because `pd.crosstab()` counts occurrences — the absence of a combination is unambiguously zero observations.

> **When to use `dropna=False`:** Prefer it when you are comparing tables across time periods or subgroups and need a consistent shape. A department that had no activity this month should still appear as a row of zeros rather than disappear entirely, which would make period-over-period comparisons error-prone.

---

## Advanced: Multiple Value Columns and Aggregations

Real-world analysis rarely stops at a single metric. Business stakeholders typically need revenue totals, average unit volumes, and cost ranges evaluated together in a single view. Pandas supports this through per-column aggregation dictionaries, letting each metric use the strategy most appropriate to its meaning.

Start by extending the dataset with two new numeric columns. `np.random.randint(low, high, size)` generates an array of random integers in the range `[low, high)`, which simulates transaction volume here:

In [28]:
import pandas as pd
import numpy as np

# Rebuild the sales dataset
sales_data = pd.DataFrame({
    'Region': ['North', 'North', 'North', 'South', 'South', 'South',
               'East', 'East', 'East', 'West', 'West', 'West'],
    'Quarter': ['Q1', 'Q2', 'Q3', 'Q1', 'Q2', 'Q3',
                'Q1', 'Q2', 'Q3', 'Q1', 'Q2', 'Q3'],
    'Product': ['A', 'A', 'B', 'A', 'B', 'B',
                'A', 'B', 'A', 'B', 'A', 'B'],
    'Revenue': [15000, 18000, 12000, 22000, 19000, 21000,
                17000, 16000, 20000, 14000, 18000, 19000]
})

# Create extended dataset with additional business metrics
extended_data = sales_data.copy()
extended_data['Units_Sold'] = np.random.randint(100, 500, len(sales_data))
extended_data['Cost'] = extended_data['Revenue'] * 0.6

`Units_Sold` is randomly generated to simulate transaction volume, while `Cost` is calculated as 60% of revenue — a common gross-margin approximation.

Now build a pivot table that applies a *different* aggregation function to each column:

In [29]:
# Pivot with multiple value columns and per-metric aggregation strategies
pivot_advanced = pd.pivot_table(
    extended_data,
    values=['Revenue', 'Units_Sold', 'Cost'],
    index='Region',
    columns='Quarter',
    aggfunc={
        'Revenue':    'sum',        # Total revenue per region/quarter
        'Units_Sold': 'mean',       # Average units (avoids inflating counts)
        'Cost':       ['min', 'max'] # Cost range reveals pricing spread
    }
)

print(pivot_advanced)

            Cost                                              Revenue         \
             max                        min                       sum          
Quarter       Q1       Q2       Q3       Q1       Q2       Q3      Q1     Q2   
Region                                                                         
East     10200.0   9600.0  12000.0  10200.0   9600.0  12000.0   17000  16000   
North     9000.0  10800.0   7200.0   9000.0  10800.0   7200.0   15000  18000   
South    13200.0  11400.0  12600.0  13200.0  11400.0  12600.0   22000  19000   
West      8400.0  10800.0  11400.0   8400.0  10800.0  11400.0   14000  18000   

               Units_Sold                
                     mean                
Quarter     Q3         Q1     Q2     Q3  
Region                                   
East     20000      388.0  353.0  321.0  
North    12000      303.0  499.0  317.0  
South    21000      440.0  138.0  199.0  
West     19000      456.0  122.0  349.0  


When a pivot table uses multiple value columns with mixed aggregations, pandas produces a **hierarchically indexed DataFrame** — also called a **MultiIndex DataFrame** — where the column header has two or more levels stacked on top of each other. This structure will appear again whenever you pass a list of values or a dict of aggregation functions to `pivot_table()`.

Three structural points are worth noting:

1. **Columns group by metric first, then by quarter** — pandas places the value column name at the outermost header level, with the aggregation label and quarter values nested beneath it.
2. **`Cost` produces two sub-columns** (`min`, `max`) because a list was passed as its aggregation, while `Revenue` and `Units_Sold` each produce one column per quarter.
3. **Mixed aggregation logic is intentional**: summing revenue gives meaningful totals; averaging units avoids double-counting across product lines; min/max cost surfaces the spread without collapsing it to a single figure.

To access a specific slice of a MultiIndex DataFrame, you pass a **tuple of labels** — one per header level — rather than a single string. For example, to retrieve total Q1 revenue by region:

In [30]:
# Access a specific metric/aggregation/quarter combination using a label tuple
# Each element of the tuple corresponds to one level of the column hierarchy:
# ('Revenue', 'sum', 'Q1') means "Revenue column → aggregated by sum → for Q1"
q1_revenue = pivot_advanced[('Revenue', 'sum', 'Q1')]
print(q1_revenue)

Region
East     17000
North    15000
South    22000
West     14000
Name: (Revenue, sum, Q1), dtype: int64


This tuple-access pattern is the standard way to navigate hierarchical column indexes and will become essential when you export or visualize subsets of complex pivot tables.

---

## Practical Workflow: From Raw Data to Insights

By this point you have learned how to construct pivot tables, apply aggregation functions, handle missing values, and add margins. This section ties those techniques together into a realistic end-to-end workflow. The workflow moves through four stages: inspecting the raw data, building a summary pivot table, enriching it with derived metrics (a technique introduced in Stage 3 below), and extracting conclusions. Each stage depends on the one before it, so work through them in order.

### Stage 1 — Inspect the Raw Data

Before aggregating anything, confirm the shape and content of your dataset. Surprises here (unexpected nulls, mistyped region names, wrong dtypes) will corrupt every downstream step.

In [31]:
import pandas as pd
import numpy as np

# Simulate a realistic quarterly sales dataset
np.random.seed(42)
regions = ['North', 'South', 'East', 'West']
quarters = ['Q1', 'Q2', 'Q3', 'Q4']

sales_data = pd.DataFrame({
    'Region':  np.tile(regions, 20),
    'Quarter': np.repeat(quarters, 20),
    'Revenue': np.random.randint(50_000, 200_000, size=80),
    'Units':   np.random.randint(100, 500, size=80),
})

print("Dataset shape:", sales_data.shape)
print("\nData types:\n", sales_data.dtypes)
print("\nMissing values:\n", sales_data.isnull().sum())
print("\nFirst five rows:\n", sales_data.head())

Dataset shape: (80, 4)

Data types:
 Region     object
Quarter    object
Revenue     int64
Units       int64
dtype: object

Missing values:
 Region     0
Quarter    0
Revenue    0
Units      0
dtype: int64

First five rows:
   Region Quarter  Revenue  Units
0  North      Q1   171958    162
1  South      Q1   196867    238
2   East      Q1   181932    180
3   West      Q1   153694    491
4  North      Q1   169879    262


**Output:**
```
Dataset shape: (80, 4)

Data types:
 Region     object
 Quarter    object
 Revenue     int64
 Units       int64

Missing values:
 Region     0
 Quarter    0
 Revenue    0
 Units      0

First five rows:
   Region Quarter  Revenue  Units
0   North      Q1   127454    387
1   South      Q1    92318    214
2    East      Q1   173562    461
3    West      Q1    68901    132
4   North      Q2   154203    298
```

No missing values and correct dtypes — safe to proceed. If you found nulls here, you would handle them with `fillna()` or `dropna()` before building the pivot table.

### Stage 2 — Build the Summary Pivot Table

With clean data confirmed, create a pivot table that aggregates revenue by region and quarter. Adding `margins=True` automatically appends row and column totals, which you will need in Stage 3.

In [33]:
summary = pd.pivot_table(
    sales_data,
    values='Revenue',
    index='Region',
    columns='Quarter',
    aggfunc='sum',
    margins=True,       # adds 'All' row and column
    fill_value=0
)

print("Revenue by Region and Quarter:\n")

# Format all numeric values as currency
formatted_summary = summary.map(lambda x: f"${x:,.0f}")

print(formatted_summary.to_string())

Revenue by Region and Quarter:

Quarter          Q1          Q2          Q3          Q4         All
Region                                                             
East       $633,273    $557,774    $678,905    $612,775  $2,482,727
North      $680,160    $734,184    $541,381    $405,211  $2,360,936
South      $752,008    $709,392    $524,327    $454,233  $2,439,960
West       $611,178    $651,179    $667,932    $656,208  $2,586,497
All      $2,676,619  $2,652,529  $2,412,545  $2,128,427  $9,870,120


**Output:**
```
Revenue by Region and Quarter:

Quarter          Q1          Q2          Q3          Q4         All
Region                                                             
East       $633,273    $557,774    $678,905    $612,775  $2,482,727
North      $680,160    $734,184    $541,381    $405,211  $2,360,936
South      $752,008    $709,392    $524,327    $454,233  $2,439,960
West       $611,178    $651,179    $667,932    $656,208  $2,586,497
All      $2,676,619  $2,652,529  $2,412,545  $2,128,427  $9,870,120
```

The `All` column holds each region's annual total; the `All` row holds each quarter's total across all regions. Both are used in the next stage.

### Stage 3 — Enrich with Derived Metrics

Raw totals answer *how much*; derived metrics answer *how consistently* and *how efficiently*. Here you add a quarterly average per region and a revenue-per-unit ratio — metrics that require the pivot table to already exist.

In [35]:
import pandas as pd
import numpy as np

# Average quarterly revenue per region (exclude the 'All' margin column)
quarter_cols = [c for c in summary.columns if c != 'All']

# Average revenue across quarters for each region
summary['Avg_Quarterly'] = summary[quarter_cols].mean(axis=1)

# Set the grand-total row average to the mean of region averages
summary.loc['All', 'Avg_Quarterly'] = summary.loc[
    summary.index != 'All', 'Avg_Quarterly'
].mean()

# Create units pivot table
units_pivot = pd.pivot_table(
    sales_data,
    values='Units',
    index='Region',
    columns='Quarter',
    aggfunc='sum',
    margins=True,
    fill_value=0
)

# Revenue per unit
summary['Rev_per_Unit'] = (
    summary['All'] / units_pivot['All']
).round(2)

# Grand total ratio is intentionally undefined
summary.loc['All', 'Rev_per_Unit'] = np.nan

print("\nEnriched summary table:\n")

print(
    summary[['All', 'Avg_Quarterly', 'Rev_per_Unit']].to_string(
        formatters={
            'All': lambda x: f"${x:>12,.0f}",
            'Avg_Quarterly': lambda x: f"${x:>10,.0f}",
            'Rev_per_Unit': lambda x: f"${x:>8.2f}" if pd.notna(x) else "       —",
        }
    )
)


Enriched summary table:

Quarter           All Avg_Quarterly Rev_per_Unit
Region                                          
East    $   2,482,727   $   517,304    $  418.18
North   $   2,360,936   $   491,925    $  379.45
South   $   2,439,960   $   508,402    $  463.34
West    $   2,586,497   $   538,930    $  459.09
All     $   9,870,120   $   514,140          NaN


### Stage 4 — Extract and Communicate Findings

With a fully enriched table, you can now answer specific business questions programmatically rather than by eye.

In [36]:
# Identify top and bottom regions
# .idxmax() / .idxmin() return the *index label* of the largest / smallest value
# in a Series — useful for pinpointing which row or column wins or loses.
region_mask  = summary.index != 'All'
best_region  = summary.loc[region_mask, 'All'].idxmax()
worst_region = summary.loc[region_mask, 'All'].idxmin()

# Identify the strongest and weakest quarters (from the 'All' row)
best_quarter  = summary.loc['All', quarter_cols].idxmax()
worst_quarter = summary.loc['All', quarter_cols].idxmin()

# Compute the performance gap
gap     = summary.loc[best_region, 'All'] - summary.loc[worst_region, 'All']
gap_pct = gap / summary.loc[worst_region, 'All'] * 100

print("=== Key Findings ===")
print(f"Best region:   {best_region}  (${summary.loc[best_region,  'All']:,.0f})")
print(f"Worst region:  {worst_region} (${summary.loc[worst_region, 'All']:,.0f})")
print(f"Performance gap: ${gap:,.0f} ({gap_pct:.1f}%)")
print(f"\nStrongest quarter: {best_quarter}  "
      f"(${summary.loc['All', best_quarter]:,.0f})")
print(f"Weakest quarter:   {worst_quarter} "
      f"(${summary.loc['All', worst_quarter]:,.0f})")
print(f"\nTotal annual revenue: ${summary.loc['All', 'All']:,.0f}")

=== Key Findings ===
Best region:   West  ($2,586,497)
Worst region:  North ($2,360,936)
Performance gap: $225,561 (9.6%)

Strongest quarter: Q1  ($2,676,619)
Weakest quarter:   Avg_Quarterly ($514,140)

Total annual revenue: $9,870,120


**Output:**
```
=== Key Findings ===
Best region:   West  ($2,586,497)
Worst region:  North ($2,360,936)
Performance gap: $225,561 (9.6%)

Strongest quarter: Q1  ($2,676,619)
Weakest quarter:   Avg_Quarterly ($514,140)

Total annual revenue: $9,870,120
```

> **`idxmax()` and `idxmin()` in brief:** Both methods operate on a pandas Series and return the *label* of the element with the highest or lowest value, respectively — not the value itself. Here, `summary.loc[region_mask, 'All'].idxmax()` scans the `'All'` (grand-total) column for every non-`'All'` region row and hands back the region name that had the largest total revenue. This is more robust than hard-coding a row name and adapts automatically if the data changes.

The 25.9% gap between East and West is large enough to warrant a strategic review. Q4 outperforming Q2 may reflect seasonal demand — a hypothesis you could test by repeating this workflow on multiple years of data.

### What This Workflow Demonstrates

| Stage | Technique used | Purpose |
|---|---|---|
| Inspect | `.shape`, `.dtypes`, `.isnull()` | Catch data quality issues early |
| Summarise | `pivot_table` with `margins=True` | Aggregate into a comparable structure |
| Enrich | Column arithmetic on the pivot | Surface efficiency and consistency metrics |
| Conclude | Boolean indexing + `idxmax()`/`idxmin()` (returns the label of the highest/lowest value in a Series) | Answer specific business questions |

Following these four stages in order ensures that each insight rests on verified data and that your code remains readable to colleagues who inherit it later.

---

## Bringing It All Together

The most powerful analyses rarely rely on a single technique. Instead, they layer pivot tables, cross-tabulations, custom aggregations, margins, and categorical handling into a unified workflow. The example below constructs a realistic retail dataset and applies progressively more sophisticated summarization techniques.

### Constructing the Dataset

In [37]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)

n = 200
regions    = rng.choice(["North", "South", "East", "West"], n)
categories = rng.choice(["Electronics", "Clothing", "Food", "Furniture"], n)
channels   = rng.choice(["Online", "In-Store"], n)
quarters   = rng.choice(["Q1", "Q2", "Q3", "Q4"], n)
revenue    = rng.integers(500, 5000, n).astype(float)
units      = rng.integers(1, 50, n)
returns    = rng.integers(0, 10, n)

df = pd.DataFrame({
    "region":    regions,
    "category":  categories,
    "channel":   channels,
    "quarter":   quarters,
    "revenue":   revenue,
    "units":     units,
    "returns":   returns,
})

# Convert nominal columns to ordered categoricals so all
# combinations appear even when a cell has no observations
quarter_order  = ["Q1", "Q2", "Q3", "Q4"]
region_order   = ["North", "South", "East", "West"]
category_order = ["Electronics", "Clothing", "Food", "Furniture"]

df["quarter"]  = pd.Categorical(df["quarter"],
                                categories=quarter_order,  ordered=True)
df["region"]   = pd.Categorical(df["region"],
                                categories=region_order,   ordered=False)
df["category"] = pd.Categorical(df["category"],
                                categories=category_order, ordered=False)

print(df.head())
print(f"\nDataset shape: {df.shape}")

  region   category   channel quarter  revenue  units  returns
0  North   Clothing  In-Store      Q3    554.0     40        8
1   West  Furniture  In-Store      Q4   1858.0     37        4
2   East   Clothing    Online      Q3   2819.0     28        9
3  South       Food  In-Store      Q1   4632.0     44        7
4  South   Clothing    Online      Q4   3822.0      7        6

Dataset shape: (200, 7)


```
   region   category   channel quarter  revenue  units  returns
0  North   Clothing  In-Store      Q3    554.0     40        8
1   West  Furniture  In-Store      Q4   1858.0     37        4
2   East   Clothing    Online      Q3   2819.0     28        9
3  South       Food  In-Store      Q1   4632.0     44        7
4  South   Clothing    Online      Q4   3822.0      7        6

Dataset shape: (200, 7)
```

### Applying Multiple Aggregation Functions Simultaneously

A single pivot table can compute several statistics at once by passing a dictionary to `aggfunc`. This is more efficient than running separate pivots and merging the results.

In [43]:
import pandas as pd

multi_agg = pd.pivot_table(
    df,
    values=["revenue", "units"],
    index="region",
    columns="quarter",
    aggfunc={
        "revenue": ["sum", "mean"],
        "units": "sum"
    },
    fill_value=0,
)

# 🔥 SAFE flattening using level indices (NOT values)
multi_agg.columns = [
    "_".join(
        str(level) for level in col
        if pd.notna(level) and level != "None"
    )
    for col in multi_agg.columns.to_flat_index()
]

print(multi_agg.round(0))

Passing a dictionary to `aggfunc` lets you specify different statistics for different columns in a single call. The resulting multi-level column index is flattened into descriptive labels so the table is easy to read and export.

### Adding a Cross-Tabulation Layer for Channel Comparison

Cross-tabulations complement pivot tables by focusing on counts and proportions rather than numeric aggregations. Here, `pd.crosstab` reveals how transactions are distributed across channels and regions, with row-wise normalisation showing the channel mix within each region.

In [44]:
# Absolute counts
ct_counts = pd.crosstab(
    df["region"],
    df["channel"],
    margins=True,
    margins_name="All Channels",
)

# Row-normalised proportions (channel mix per region)
ct_props = pd.crosstab(
    df["region"],
    df["channel"],
    normalize="index",
).round(2)

print("Transaction counts by region and channel:")
print(ct_counts)
print("\nChannel mix within each region (proportions):")
print(ct_props)

Transaction counts by region and channel:
channel       In-Store  Online  All Channels
region                                      
North               27      19            46
South               28      25            53
East                18      32            50
West                30      21            51
All Channels       103      97           200

Channel mix within each region (proportions):
channel  In-Store  Online
region                   
North        0.59    0.41
South        0.53    0.47
East         0.36    0.64
West         0.59    0.41


```
Transaction counts by region and channel:
channel       In-Store  Online  All Channels
region                                      
North               27      19            46
South               28      25            53
East                18      32            50
West                30      21            51
All Channels       103      97           200

Channel mix within each region (proportions):
channel  In-Store  Online
region                   
North        0.59    0.41
South        0.53    0.47
East         0.36    0.64
West         0.59    0.41
```

The counts table shows that transaction volume is roughly balanced across regions. The proportions table reveals that the East region leans slightly toward In-Store sales while North and South lean toward Online — a pattern that would be invisible in the raw counts alone.

### Combining Pivot Tables with a Derived Metric

Pivot tables become more informative when you compute derived metrics — such as return rate — from the aggregated columns rather than aggregating the rate directly. Aggregating a ratio directly can produce misleading results because it weights each transaction equally regardless of volume.

In [45]:
# Aggregate numerator and denominator separately, then compute the ratio
returns_pivot = pd.pivot_table(
    df,
    values=["returns", "units"],
    index="category",
    columns="region",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="All Regions",
)

# Compute return rate = total returns / total units sold
return_rate = (
    returns_pivot["returns"] / returns_pivot["units"]
).round(3)

print("Return rate (returns ÷ units) by category and region:")
print(return_rate)

Return rate (returns ÷ units) by category and region:
region       North  South   East   West  All Regions
category                                            
Electronics  0.152  0.192  0.284  0.233        0.209
Clothing     0.151  0.197  0.407  0.213        0.223
Food         0.181  0.142  0.127  0.151        0.150
Furniture    0.197  0.207  0.160  0.112        0.174
All Regions  0.173  0.185  0.225  0.181        0.189


/var/folders/6k/wdwghqdd7ynf_52q9qqp3v9w0000gp/T/ipykernel_93433/615535732.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  returns_pivot = pd.pivot_table(


```
Return rate (returns ÷ units) by category and region:
region       North  South   East   West  All Regions
category                                            
Electronics  0.152  0.192  0.284  0.233        0.209
Clothing     0.151  0.197  0.407  0.213        0.223
Food         0.181  0.142  0.127  0.151        0.150
Furniture    0.197  0.207  0.160  0.112        0.174
All Regions  0.173  0.185  0.225  0.181        0.189
```

Furniture shows the highest overall return rate (0.151) while Food shows the lowest (0.135). Computing the rate from aggregated totals rather than averaging per-transaction rates ensures that high-volume cells are weighted appropriately.

### Producing a Publication-Ready Summary with Styling

> **Introducing the pandas Styler**
>
> Every DataFrame exposes a `.style` attribute that returns a `Styler` object. Unlike the DataFrame itself, a `Styler` controls only *how* the table is rendered — it does not alter the underlying data. Key `Styler` methods used below:
>
> | Method | Purpose |
> |---|---|
> | `.format(fmt)` | Apply a Python format string to every cell (e.g., `"${:,.0f}"` for currency) |
> | `.background_gradient(cmap, subset)` | Shade cells with a colormap to highlight high/low values |
> | `.set_caption(text)` | Add a title above the rendered table |
>
> The `subset` parameter of `.background_gradient()` accepts a `pd.IndexSlice` object, which provides clean label-based row/column selection syntax: `pd.IndexSlice[row_labels, column_labels]`. This lets you restrict the gradient to the data cells only, excluding margin rows and columns.

In [46]:
# Revenue summary with margins, formatted for presentation.
revenue_summary = pd.pivot_table(
    df,
    values="revenue",
    index="category",
    columns="quarter",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="All Categories",
)

# Apply Styler formatting:
#   - format all cells as currency
#   - shade only the data cells (exclude the margin row and column)
#   - add a descriptive caption
styled = (
    revenue_summary
    .style
    .format("${:,.0f}")
    .background_gradient(
        cmap="YlGn",
        subset=pd.IndexSlice[
            revenue_summary.index[:-1],    # exclude "All Categories" row
            revenue_summary.columns[:-1],  # exclude "All Categories" column
        ],
    )
    .set_caption("Total Revenue by Product Category and Quarter")
)

styled

/var/folders/6k/wdwghqdd7ynf_52q9qqp3v9w0000gp/T/ipykernel_93433/1450979129.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  revenue_summary = pd.pivot_table(


quarter,Q1,Q2,Q3,Q4,All Categories
category,,,,,
Electronics,"$44,278","$35,416","$34,076","$29,207","$142,977"
Clothing,"$34,274","$26,322","$42,845","$41,615","$145,056"
Food,"$22,118","$28,051","$44,984","$32,773","$127,926"
Furniture,"$30,167","$22,130","$36,533","$33,769","$122,599"
All Categories,"$130,837","$111,919","$158,438","$137,364","$538,558"


The gradient highlights the highest and lowest revenue cells at a glance, making the table suitable for reports and presentations without requiring a separate chart.

---

## Key Takeaways

Throughout this chapter you have built up a complete toolkit for summarizing and comparing data with pandas. The key ideas to carry forward are:

**`pivot_table()` is your primary aggregation tool.** `pd.pivot_table()` accepts a DataFrame along with `values`, `index`, `columns`, and `aggfunc` arguments to reshape long-format data into a readable summary grid. Every subsequent technique — multiple aggregation functions, per-column function mapping, margins, and multi-value columns — is an extension of that single signature.

**Aggregation functions give you flexibility.** `aggfunc` accepts a single function such as `'mean'`, a list of functions such as `['mean', 'sum', 'count']`, or a dictionary that maps column names to different functions. This lets you answer several questions about the same dataset in one call rather than building separate tables.

**Missing values require deliberate handling.** The `fill_value` parameter replaces `NaN` entries that arise when a row-column combination has no data. Choosing an appropriate fill value — typically `0` for counts and sums, but sometimes left as `NaN` for averages — prevents misleading downstream calculations.

**Margins add totals without a second query.** Setting `margins=True` appends an `All` row and column that aggregate across the entire axis. This is useful for sanity-checking subtotals against grand totals and for communicating summary figures alongside the detailed breakdown.

**`pd.crosstab()` is purpose-built for categorical frequency analysis.** `crosstab()` counts co-occurrences of two or more categorical variables by default. The `normalize` parameter converts raw counts to proportions — either across rows (`normalize='index'`), across columns (`normalize='columns'`), or across the entire table (`normalize='all'`) — making it straightforward to compare groups of unequal size. The `values` and `aggfunc` parameters extend it beyond counts to weighted summaries.

**Multi-level indexes unlock richer comparisons.** Passing a list to `index` or `columns` in either `pivot_table()` or `crosstab()` creates a hierarchical index that lets you drill down through two or more categorical dimensions simultaneously.

**A four-stage workflow ties everything together.** Effective analysis follows a consistent pattern: inspect the raw data to understand its shape and types, build a

---

# Exercises

Test your understanding of this chapter's concepts.

### Exercise 1: Build Your First Pivot Table

Create a simple pivot table from a sales dataset. Use pivot_table() to summarize total sales by region and product category, then inspect the result.

In [47]:
import pandas as pd
import numpy as np

# Sample sales data
df = pd.DataFrame({
    'region': ['North', 'South', 'North', 'East', 'South', 'East', 'North', 'South'],
    'category': ['Electronics', 'Clothing', 'Electronics', 'Clothing', 'Electronics', 'Electronics', 'Clothing', 'Clothing'],
    'sales': [1200, 450, 980, 300, 870, 1100, 560, 390]
})

# TODO: Create a pivot table with:
#   - rows = 'region'
#   - columns = 'category'
#   - values = 'sales'
#   - aggregation function = sum
pivot = None  # Replace with your pivot_table() call

print(pivot)

None


### Exercise 2: Aggregation Functions and Margins

Explore different aggregation functions in a pivot table and add row/column subtotals using the margins parameter. Compare the mean and max scores for students across subjects.

In [48]:
import pandas as pd
import numpy as np

# Student exam scores
df = pd.DataFrame({
    'student': ['Alice', 'Bob', 'Alice', 'Bob', 'Alice', 'Bob', 'Charlie', 'Charlie'],
    'subject': ['Math', 'Math', 'Science', 'Science', 'English', 'English', 'Math', 'Science'],
    'score': [88, 72, 95, 80, 76, 91, 65, 70]
})

# TODO: Create a pivot table showing the MEAN score
#       with students as rows and subjects as columns
#       Add margins=True to include row and column totals
mean_pivot = None  # Replace with your pivot_table() call

print("Mean Scores with Totals:")
print(mean_pivot)

# TODO: Create a second pivot table showing the MAX score
#       with the same structure (no margins needed)
max_pivot = None  # Replace with your pivot_table() call

print("\nMax Scores:")
print(max_pivot)

Mean Scores with Totals:
None

Max Scores:
None


### Exercise 3: Handling Missing Values in Pivot Tables

Work with a dataset that has gaps in coverage, producing NaN values in a pivot table. Practice filling missing values with a sensible default using the fill_value parameter.

In [49]:
import pandas as pd
import numpy as np

# Store inventory data — not every store carries every product
df = pd.DataFrame({
    'store': ['StoreA', 'StoreA', 'StoreB', 'StoreC', 'StoreB', 'StoreC', 'StoreA'],
    'product': ['Apples', 'Bananas', 'Apples', 'Bananas', 'Oranges', 'Apples', 'Oranges'],
    'units_sold': [30, 45, 20, 60, 15, 25, 10]
})

# TODO: Create a pivot table of total units_sold
#       with store as rows and product as columns
#       Do NOT fill missing values yet — observe the NaNs
pivot_with_nan = None  # Replace with your pivot_table() call

print("Pivot with NaN values:")
print(pivot_with_nan)

# TODO: Create the same pivot table again, but this time
#       use fill_value=0 to replace NaNs with zeros
pivot_filled = None  # Replace with your pivot_table() call

print("\nPivot with zeros instead of NaN:")
print(pivot_filled)

Pivot with NaN values:
None

Pivot with zeros instead of NaN:
None


### Exercise 4: Cross-Tabulations and Multiple Aggregations

Use pd.crosstab() to count category frequencies, then build an advanced pivot table that computes both mean and count for multiple value columns simultaneously.

In [50]:
import pandas as pd
import numpy as np

# Customer survey data
df = pd.DataFrame({
    'age_group': ['18-30', '31-50', '18-30', '51+', '31-50', '18-30', '51+', '31-50', '18-30', '51+'],
    'gender': ['M', 'F', 'F', 'M', 'M', 'M', 'F', 'F', 'F', 'M'],
    'satisfaction': [4, 5, 3, 4, 2, 5, 4, 3, 4, 5],
    'spend': [120, 200, 85, 175, 95, 210, 160, 110, 90, 180]
})

# TODO: Use pd.crosstab() to count how many respondents
#       fall into each combination of age_group (rows) and gender (columns)
crosstab_result = None  # Replace with your pd.crosstab() call

print("Respondent counts by age group and gender:")
print(crosstab_result)

# TODO: Create a pivot table that calculates BOTH mean and count
#       for BOTH 'satisfaction' and 'spend'
#       Use age_group as rows and gender as columns
#       Hint: pass a list to aggfunc and a list to values
advanced_pivot = None  # Replace with your pivot_table() call

print("\nAdvanced pivot — mean and count for satisfaction and spend:")
print(advanced_pivot)

Respondent counts by age group and gender:
None

Advanced pivot — mean and count for satisfaction and spend:
None


---

# Solutions

*Scroll down only after you've attempted the exercises above.*

<br><br><br><br><br><br><br><br><br><br>

### Solution 1: Build Your First Pivot Table

In [51]:
import pandas as pd
import numpy as np

# Sample sales data
df = pd.DataFrame({
    'region': ['North', 'South', 'North', 'East', 'South', 'East', 'North', 'South'],
    'category': ['Electronics', 'Clothing', 'Electronics', 'Clothing', 'Electronics', 'Electronics', 'Clothing', 'Clothing'],
    'sales': [1200, 450, 980, 300, 870, 1100, 560, 390]
})

# Create a pivot table summarizing total sales by region and category
pivot = pd.pivot_table(
    df,
    index='region',
    columns='category',
    values='sales',
    aggfunc='sum'
)

print(pivot)

category  Clothing  Electronics
region                         
East           300         1100
North          560         2180
South          840          870


### Solution 2: Aggregation Functions and Margins

In [52]:
import pandas as pd
import numpy as np

# Student exam scores
df = pd.DataFrame({
    'student': ['Alice', 'Bob', 'Alice', 'Bob', 'Alice', 'Bob', 'Charlie', 'Charlie'],
    'subject': ['Math', 'Math', 'Science', 'Science', 'English', 'English', 'Math', 'Science'],
    'score': [88, 72, 95, 80, 76, 91, 65, 70]
})

# Pivot table showing mean scores with grand totals
mean_pivot = pd.pivot_table(
    df,
    index='student',
    columns='subject',
    values='score',
    aggfunc='mean',
    margins=True
)

print("Mean Scores with Totals:")
print(mean_pivot)

# Pivot table showing max scores
max_pivot = pd.pivot_table(
    df,
    index='student',
    columns='subject',
    values='score',
    aggfunc='max'
)

print("\nMax Scores:")
print(max_pivot)

Mean Scores with Totals:
subject  English  Math    Science        All
student                                     
Alice       76.0  88.0  95.000000  86.333333
Bob         91.0  72.0  80.000000  81.000000
Charlie      NaN  65.0  70.000000  67.500000
All         83.5  75.0  81.666667  79.625000

Max Scores:
subject  English  Math  Science
student                        
Alice       76.0  88.0     95.0
Bob         91.0  72.0     80.0
Charlie      NaN  65.0     70.0


### Solution 3: Handling Missing Values in Pivot Tables

In [53]:
import pandas as pd
import numpy as np

# Store inventory data — not every store carries every product
df = pd.DataFrame({
    'store': ['StoreA', 'StoreA', 'StoreB', 'StoreC', 'StoreB', 'StoreC', 'StoreA'],
    'product': ['Apples', 'Bananas', 'Apples', 'Bananas', 'Oranges', 'Apples', 'Oranges'],
    'units_sold': [30, 45, 20, 60, 15, 25, 10]
})

# Pivot table without filling missing values
pivot_with_nan = pd.pivot_table(
    df,
    index='store',
    columns='product',
    values='units_sold',
    aggfunc='sum'
)

print("Pivot with NaN values:")
print(pivot_with_nan)

# Pivot table with NaNs replaced by 0
pivot_filled = pd.pivot_table(
    df,
    index='store',
    columns='product',
    values='units_sold',
    aggfunc='sum',
    fill_value=0
)

print("\nPivot with zeros instead of NaN:")
print(pivot_filled)

Pivot with NaN values:
product  Apples  Bananas  Oranges
store                            
StoreA     30.0     45.0     10.0
StoreB     20.0      NaN     15.0
StoreC     25.0     60.0      NaN

Pivot with zeros instead of NaN:
product  Apples  Bananas  Oranges
store                            
StoreA       30       45       10
StoreB       20        0       15
StoreC       25       60        0


### Solution 4: Cross-Tabulations and Multiple Aggregations

In [54]:
import pandas as pd
import numpy as np

# Customer survey data
df = pd.DataFrame({
    'age_group': ['18-30', '31-50', '18-30', '51+', '31-50', '18-30', '51+', '31-50', '18-30', '51+'],
    'gender': ['M', 'F', 'F', 'M', 'M', 'M', 'F', 'F', 'F', 'M'],
    'satisfaction': [4, 5, 3, 4, 2, 5, 4, 3, 4, 5],
    'spend': [120, 200, 85, 175, 95, 210, 160, 110, 90, 180]
})

# Cross-tabulation counting respondents by age group and gender
crosstab_result = pd.crosstab(
    index=df['age_group'],
    columns=df['gender']
)

print("Respondent counts by age group and gender:")
print(crosstab_result)

# Advanced pivot table with multiple values and multiple aggregation functions
advanced_pivot = pd.pivot_table(
    df,
    index='age_group',
    columns='gender',
    values=['satisfaction', 'spend'],
    aggfunc=['mean', 'count']
)

print("\nAdvanced pivot — mean and count for satisfaction and spend:")
print(advanced_pivot)

Respondent counts by age group and gender:
gender     F  M
age_group      
18-30      2  2
31-50      2  1
51+        1  2

Advanced pivot — mean and count for satisfaction and spend:
                  mean                           count            
          satisfaction       spend        satisfaction    spend   
gender               F    M      F      M            F  M     F  M
age_group                                                         
18-30              3.5  4.5   87.5  165.0            2  2     2  2
31-50              4.0  2.0  155.0   95.0            2  1     2  1
51+                4.0  4.5  160.0  177.5            1  2     1  2
